# RepoGuard Security Scanner

Test RepoGuard on vulnerable code samples.

## Install Dependencies

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'requests', 'transformers', 'torch', 'sentence-transformers'], check=True)
print('Dependencies installed')

## Setup

In [ ]:
import sys
sys.path.insert(0, '/home/a/Desktop/repoguard')

from repoguard import RepoGuard
from repoguard.engine.aggregator import Reporter
print('RepoGuard imported successfully')

## Test 1: Basic Scan

In [ ]:
guard = RepoGuard(use_llm=False)
report = guard.scan_file('/home/a/Desktop/vuln_app.py')

reporter = Reporter()
reporter.print_report(report)
print('Total findings:', report['summary']['total_findings'])

## Test 2: Scan with TinyLlama LLM

In [ ]:
guard = RepoGuard(use_llm=True, llm_model='tinyllama:1.1b')
report = guard.scan_file('/home/a/Desktop/vuln_app.py')

reporter = Reporter()
reporter.print_report(report)

## Test 3: RL Learner (Train on Findings)

In [ ]:
from repoguard.learner import LearnerLLM, run_rl_cycle

# Test code with vulnerabilities
test_code = '''
import os
import sqlite3

API_KEY = 'sk-1234567890abcdef'

def get_user(user_id):
    conn = sqlite3.connect('app.db')
    c = conn.cursor()
    query = f"SELECT * FROM users WHERE id = {user_id}"
    c.execute(query)
    return c.fetchone()

def run_cmd(cmd):
    return os.system(cmd)
'''

# Run RL cycle
results = run_rl_cycle(test_code, iterations=2)
print('Findings history:', results['findings_history'])
print('Learner trained:', results['model'].is_trained)

## Test 4: JSON Output

In [ ]:
import json
guard = RepoGuard(use_llm=False)
report = guard.scan_file('/home/a/Desktop/vuln_app.py')
print(json.dumps(report, indent=2)[:1000])